# 02 — Market Intelligence: 10 Business SQL Queries

Using `pandasql` to run SQL on in-memory Google Trends data.

**Dataset:** Real Google Trends (pytrends API) — 14 keywords, 5 years, worldwide + US regional.


In [1]:
import pandas as pd
import pandasql as ps
import numpy as np
from datetime import datetime

df_ww = pd.read_csv('data/interest_over_time_worldwide.csv', index_col=0, parse_dates=True).reset_index()
df_ww.columns = ['date'] + list(df_ww.columns[1:])
df_us = pd.read_csv('data/interest_over_time_us.csv', index_col=0, parse_dates=True).reset_index()
df_us.columns = ['date'] + list(df_us.columns[1:])
df_region = pd.read_csv('data/interest_by_region_us.csv')

# Melt to long format for SQL
ww_long = df_ww.melt(id_vars=['date'], var_name='keyword', value_name='interest')
us_long = df_us.melt(id_vars=['date'], var_name='keyword', value_name='interest')

print(f'ww_long: {ww_long.shape}, us_long: {us_long.shape}, region: {df_region.shape}')

ww_long: (3668, 3), us_long: (3668, 3), region: (714, 6)


## Q1 — Topic Interest Ranking by Volume & Growth Rate

In [2]:
q1 = '''
SELECT 
    keyword,
    ROUND(AVG(interest), 2) as avg_interest,
    ROUND(MAX(interest), 2) as peak_interest,
    ROUND(AVG(CASE WHEN date >= date('now', '-1 year') THEN interest END), 2) as recent_avg,
    ROUND(AVG(CASE WHEN date >= date('now', '-2 year') AND date < date('now', '-1 year') THEN interest END), 2) as prior_avg
FROM ww_long
GROUP BY keyword
ORDER BY avg_interest DESC
'''
print(ps.sqldf(q1, locals()).to_string(index=False))

         keyword  avg_interest  peak_interest  recent_avg  prior_avg
          Amazon         74.99          100.0       69.81      69.10
          crypto         34.40          100.0       45.67      31.52
         Netflix         31.37           44.0       29.31      29.75
              AI         28.06          100.0       67.58      35.21
         ChatGPT         25.88           82.0       68.88      37.96
    stock market         19.36           93.0       29.56      21.19
       inflation         14.77           37.0       22.50      11.46
         fitness          7.54           14.0        9.31       7.13
         Bitcoin          6.91           24.0        7.37       6.56
           Tesla          6.03           14.0        6.90       6.69
       recession          3.08           17.0        3.21       2.67
   mental health          2.24            5.0        3.37       2.00
machine learning          0.89            3.0        1.35       0.96
      telehealth          0.04    

## Q2 — Regional Interest Heatmap (top state per topic)

In [3]:
q2 = '''
SELECT keyword, geoName, interest
FROM (
    SELECT keyword, geoName, interest,
           ROW_NUMBER() OVER (PARTITION BY keyword ORDER BY interest DESC) as rn
    FROM df_region
)
WHERE rn = 1
ORDER BY interest DESC
'''
print(ps.sqldf(q2, locals()).to_string(index=False))

         keyword              geoName  interest
              AI              Wyoming       100
          Amazon           Washington       100
         Bitcoin               Hawaii       100
         ChatGPT District of Columbia       100
         Netflix        Massachusetts       100
           Tesla           California       100
          crypto              Wyoming       100
         fitness         Rhode Island       100
       inflation District of Columbia       100
machine learning              Wyoming       100
   mental health              Wyoming       100
       recession District of Columbia       100
    stock market              Wyoming       100
      telehealth              Wyoming       100


## Q3 — Emerging Topic Detection (growth >100% YoY)

In [4]:
# Compute YoY growth in a temp table
ww_long['year'] = ww_long['date'].dt.year
yearly = ww_long.groupby(['keyword', 'year'])['interest'].mean().reset_index()
yearly['prior'] = yearly.groupby('keyword')['interest'].shift(1)
yearly['growth_pct'] = ((yearly['interest'] - yearly['prior']) / yearly['prior'] * 100).round(1)

q3 = '''
SELECT keyword, year, ROUND(interest, 2) as avg_interest, growth_pct
FROM yearly
WHERE growth_pct > 100 AND year >= 2022
ORDER BY growth_pct DESC
LIMIT 15
'''
print(ps.sqldf(q3, locals()).to_string(index=False))

         keyword  year  avg_interest  growth_pct
         ChatGPT  2022          0.29         inf
machine learning  2022          0.85         inf
      telehealth  2025          0.04         inf
         ChatGPT  2023         15.62      5315.8
      telehealth  2026          0.42       994.7
       recession  2022          5.19       419.2
              AI  2023         19.30       219.6
         ChatGPT  2025         63.73       142.6


## Q4 — Trend Correlation Matrix (SQL pivot)

In [5]:
# Self-join to compute correlations
corr_pairs = []
keywords = ww_long['keyword'].unique()
for i, a in enumerate(keywords):
    for b in keywords[i+1:]:
        merged = ww_long[ww_long['keyword'] == a][['date', 'interest']].merge(
            ww_long[ww_long['keyword'] == b][['date', 'interest']], on='date', suffixes=('_a', '_b')
        )
        if len(merged) > 10:
            r = np.corrcoef(merged['interest_a'], merged['interest_b'])[0, 1]
            corr_pairs.append({'topic_a': a, 'topic_b': b, 'correlation': round(r, 3)})
corr_df = pd.DataFrame(corr_pairs)
print(corr_df.sort_values('correlation', ascending=False).head(15).to_string(index=False))

         topic_a          topic_b  correlation
              AI          ChatGPT        0.957
   mental health        inflation        0.796
              AI    mental health        0.760
         Bitcoin           crypto        0.703
         ChatGPT    mental health        0.653
   mental health          fitness        0.631
              AI          fitness        0.625
              AI     stock market        0.616
         ChatGPT     stock market        0.616
machine learning    mental health        0.608
              AI machine learning        0.607
              AI        inflation        0.606
      telehealth    mental health        0.579
machine learning        inflation        0.578
   mental health     stock market        0.572


## Q5 — Seasonal Pattern Detection (monthly aggregation)

In [6]:
ww_long['month'] = ww_long['date'].dt.month
seasonal = ww_long.groupby(['keyword', 'month'])['interest'].mean().reset_index()

q5 = '''
SELECT keyword, month, ROUND(interest, 2) as avg_interest
FROM seasonal
WHERE keyword IN ('fitness', 'mental health', 'crypto', 'AI')
ORDER BY keyword, month
'''
print(ps.sqldf(q5, locals()).to_string(index=False))

      keyword  month  avg_interest
           AI      1         27.09
           AI      2         31.55
           AI      3         36.43
           AI      4         35.48
           AI      5         24.17
           AI      6         23.09
           AI      7         20.55
           AI      8         25.41
           AI      9         30.14
           AI     10         25.87
           AI     11         30.52
           AI     12         27.18
       crypto      1         39.41
       crypto      2         34.85
       crypto      3         35.87
       crypto      4         29.62
       crypto      5         36.96
       crypto      6         25.95
       crypto      7         25.50
       crypto      8         35.77
       crypto      9         32.14
       crypto     10         34.57
       crypto     11         44.29
       crypto     12         37.86
      fitness      1          8.64
      fitness      2          7.95
      fitness      3          7.87
      fitness      4

## Q6 — Event-Driven Spike Analysis (weeks with >3σ moves)

In [7]:
# Z-score based spike detection
spike_data = []
for kw in ww_long['keyword'].unique():
    sub = ww_long[ww_long['keyword'] == kw].sort_values('date')
    sub['rolling_mean'] = sub['interest'].rolling(12).mean()
    sub['rolling_std'] = sub['interest'].rolling(12).std()
    sub['zscore'] = (sub['interest'] - sub['rolling_mean']) / sub['rolling_std']
    spikes = sub[sub['zscore'] > 3]
    for _, row in spikes.iterrows():
        spike_data.append({'keyword': kw, 'date': row['date'].strftime('%Y-%m-%d'), 'interest': row['interest'], 'zscore': round(row['zscore'], 2)})
spike_df = pd.DataFrame(spike_data)
print(spike_df.head(15).to_string(index=False) if len(spike_df) > 0 else 'No extreme spikes (>3σ) detected in this window.')

         keyword       date  interest  zscore
              AI 2022-01-02         6    3.18
machine learning 2022-01-23         1    3.18
machine learning 2025-08-10         2    3.18
machine learning 2026-01-25         2    3.18
         ChatGPT 2022-12-04         4    3.18
           Tesla 2021-08-22         6    3.18
           Tesla 2025-03-09        14    3.00
         Netflix 2024-11-10        41    3.00
          Amazon 2024-07-14        92    3.13
          Amazon 2025-07-06        81    3.03
      telehealth 2025-09-21         1    3.18
      telehealth 2026-02-22         1    3.18
   mental health 2022-10-09         3    3.18
   mental health 2023-10-08         3    3.18
   mental health 2024-10-06         3    3.18


## Q7 — Category Lifecycle: Early Growth → Peak → Decline

In [8]:
lifecycle = []
for kw in ww_long['keyword'].unique():
    sub = ww_long[ww_long['keyword'] == kw].sort_values('date')
    peak_idx = sub['interest'].idxmax()
    peak_date = sub.loc[peak_idx, 'date']
    peak_val = sub.loc[peak_idx, 'interest']
    early = sub[sub['date'] < peak_date]['interest'].mean() if (sub['date'] < peak_date).any() else 0
    late = sub[sub['date'] > peak_date]['interest'].mean() if (sub['date'] > peak_date).any() else 0
    lifecycle.append({'keyword': kw, 'peak_date': peak_date.strftime('%Y-%m-%d'), 'peak_interest': peak_val,
                      'early_avg': round(early, 1), 'late_avg': round(late, 1),
                      'trend': 'Rising' if late > early * 1.1 else ('Declining' if late < early * 0.9 else 'Stable')})
lifecycle_df = pd.DataFrame(lifecycle).sort_values('peak_date')
print(lifecycle_df.to_string(index=False))

         keyword  peak_date  peak_interest  early_avg  late_avg     trend
          crypto 2021-05-09            100        0.0      34.1    Rising
         Bitcoin 2021-05-16             24       14.0       6.8 Declining
         Netflix 2022-01-02             44       33.1      31.1    Stable
       recession 2022-06-12             17        2.0       3.3    Rising
          Amazon 2023-11-26            100       79.4      70.2 Declining
           Tesla 2025-03-09             14        5.6       7.1    Rising
    stock market 2025-04-06             93       16.3      29.1    Rising
              AI 2025-09-14            100       20.9      73.5    Rising
         ChatGPT 2025-09-14             82       19.2      69.0    Rising
      telehealth 2025-09-21              1        0.0       0.3    Rising
   mental health 2026-02-15              5        2.1       4.2    Rising
machine learning 2026-03-22              3        0.9       2.0    Rising
       inflation 2026-03-22           

## Q8 — Cross-Category Opportunity (low competition, high growth)

In [9]:
# Compute mean (proxy for competition) and recent YoY growth
opp = []
for kw in ww_long['keyword'].unique():
    sub = ww_long[ww_long['keyword'] == kw].sort_values('date')
    mean_i = sub['interest'].mean()
    recent = sub['interest'].iloc[-52:].mean()
    prior = sub['interest'].iloc[-104:-52].mean() if len(sub) >= 104 else sub['interest'].iloc[:52].mean()
    growth = ((recent - prior) / prior * 100) if prior > 0 else 0
    opp.append({'keyword': kw, 'mean_interest': round(mean_i, 1), 'recent_growth_pct': round(growth, 1)})
opp_df = pd.DataFrame(opp)
opp_df['opportunity_score'] = (opp_df['recent_growth_pct'] / (opp_df['mean_interest'] + 1)).round(2)
print(opp_df.sort_values('opportunity_score', ascending=False).to_string(index=False))

         keyword  mean_interest  recent_growth_pct  opportunity_score
   mental health            2.2               68.3              21.34
machine learning            0.9               40.0              21.05
       inflation           14.8               96.3               6.09
       recession            3.1               20.1               4.90
         fitness            7.5               30.5               3.59
              AI           28.1               91.9               3.16
         ChatGPT           25.9               81.5               3.03
    stock market           19.4               39.5               1.94
         Bitcoin            6.9               12.3               1.56
          crypto           34.4               44.9               1.27
           Tesla            6.0                3.2               0.46
          Amazon           75.0                1.0               0.01
      telehealth            0.0                0.0               0.00
         Netflix    

## Q9 — Interest Forecasting (Simple Linear Projection)

In [10]:
from sklearn.linear_model import LinearRegression

forecast_results = []
for kw in ww_long['keyword'].unique():
    sub = ww_long[ww_long['keyword'] == kw].sort_values('date').dropna()
    X = np.arange(len(sub)).reshape(-1, 1)
    y = sub['interest'].values
    model = LinearRegression().fit(X, y)
    future_X = np.array([[len(sub) + i] for i in range(1, 53)])  # 1 year ahead
    preds = model.predict(future_X)
    trend_slope = model.coef_[0]
    forecast_results.append({'keyword': kw, 'trend_slope': round(trend_slope, 3),
                             'forecast_next_yr_avg': round(preds.mean(), 1),
                             'trend_direction': 'Up' if trend_slope > 0.05 else ('Down' if trend_slope < -0.05 else 'Flat')})
forecast_df = pd.DataFrame(forecast_results).sort_values('trend_slope', ascending=False)
print(forecast_df.to_string(index=False))

         keyword  trend_slope  forecast_next_yr_avg trend_direction
         ChatGPT        0.322                  76.7              Up
              AI        0.288                  73.5              Up
    stock market        0.061                  29.1              Up
       inflation        0.035                  20.2            Flat
          crypto        0.009                  35.9            Flat
           Tesla        0.008                   7.3            Flat
         fitness        0.008                   8.8            Flat
   mental health        0.006                   3.1            Flat
machine learning        0.004                   1.6            Flat
      telehealth        0.001                   0.2            Flat
       recession        0.001                   3.2            Flat
         Bitcoin       -0.005                   6.1            Flat
         Netflix       -0.022                  27.8            Flat
          Amazon       -0.060                  6

## Q10 — Geographic Arbitrage (topics hot in one state, cold in another)

In [11]:
# For each keyword, find max state and min state (with non-zero interest)
arbitrage = []
for kw in df_region['keyword'].unique():
    sub = df_region[(df_region['keyword'] == kw) & (df_region['interest'] > 0)]
    if len(sub) > 1:
        max_row = sub.loc[sub['interest'].idxmax()]
        min_row = sub.loc[sub['interest'].idxmin()]
        ratio = max_row['interest'] / min_row['interest'] if min_row['interest'] > 0 else float('inf')
        arbitrage.append({'keyword': kw, 'hottest_state': max_row['geoName'], 'hottest_interest': max_row['interest'],
                          'coldest_state': min_row['geoName'], 'coldest_interest': min_row['interest'],
                          'arbitrage_ratio': round(ratio, 1)})
arb_df = pd.DataFrame(arbitrage).sort_values('arbitrage_ratio', ascending=False)
print(arb_df.to_string(index=False))

         keyword        hottest_state  hottest_interest        coldest_state  coldest_interest  arbitrage_ratio
machine learning              Wyoming               100                Maine                 8             12.5
           Tesla           California               100          Mississippi                24              4.2
       inflation District of Columbia               100          Mississippi                31              3.2
      telehealth              Wyoming               100              Indiana                36              2.8
              AI              Wyoming               100         South Dakota                38              2.6
         Bitcoin               Hawaii               100          Mississippi                42              2.4
    stock market              Wyoming               100          Mississippi                41              2.4
          crypto              Wyoming               100             Kentucky                42          